In [3]:
%pip install -qU pandas tqdm

from tqdm import tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Utils


In [9]:
%pip install -q newspaper3k lxml

from newspaper import Article


def fetch_one_article(url: str) -> str | None:
    try:
        article = Article(url)
        article.download()
        article.parse()
        return article.text
    except Exception as e:
        print(f"Error fetching {url}: {str(e)}")
        return None



[notice] A new release of pip is available: 24.0 -> 24.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [5]:
from urllib.parse import urlparse


def get_domain(url):
    return urlparse(url).netloc.split("www.")[-1]


### Load not fetchable domains

Domains that we know can't be fetched using newspaper3k because of restrictions


In [8]:
NOT_FETCHABLE_DOMAINS = open("not_fechable_domains.txt").read().splitlines()

NOT_FETCHABLE_DOMAINS[:3]


['skift.com', '247wallst.com', 'windowsreport.com']

# Update MongoDB with article contents


In [2]:
from dotenv import load_dotenv
from pymongo import MongoClient
import os

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))

db = client["blogdb"]

collection = db["ai_news"]


### Add a domain field


In [6]:
from pymongo import UpdateOne

# Batch size for processing
BATCH_SIZE = 1000

# Find all documents without a domain field
docs_without_domain = collection.find(
    {"domain": {"$exists": False}}, {"_id": 1, "url": 1}
)

# Count total documents to update for the progress bar
total_docs = collection.count_documents({"domain": {"$exists": False}})

updated_count = 0
bulk_operations = []

# Process documents in batches
for doc in tqdm(docs_without_domain, total=total_docs, desc="Processing documents"):
    if "url" in doc:
        domain = get_domain(doc["url"])
        bulk_operations.append(
            UpdateOne({"_id": doc["_id"]}, {"$set": {"domain": domain}})
        )

    # If we've reached the batch size, execute the bulk operation
    if len(bulk_operations) >= BATCH_SIZE:
        result = collection.bulk_write(bulk_operations)
        updated_count += result.modified_count
        bulk_operations = []

# Execute any remaining operations
if bulk_operations:
    result = collection.bulk_write(bulk_operations)
    updated_count += result.modified_count

print(f"Added domain to {updated_count} documents")

# Verify that all documents now have a domain
remaining_docs = collection.count_documents({"domain": {"$exists": False}})
assert (
    remaining_docs == 0
), f"There are still {remaining_docs} documents without a domain"


Processing documents: 100%|██████████| 26106/26106 [01:30<00:00, 289.35it/s]


Added domain to 26106 documents


In [ ]:
import time
import random
from tqdm import tqdm
from collections import defaultdict


def process_articles():
    # Query for documents without page_content and with fetchable domains
    query = {
        "page_content": {"$exists": False},
        "domain": {"$nin": NOT_FETCHABLE_DOMAINS},
    }

    projection = {"_id": 1, "url": 1}

    # Get total count for progress bar
    total_docs = collection.count_documents(query)

    previous_domain = None
    domain_errors = defaultdict(int)
    ignored_domains = set()

    for doc in tqdm(
        collection.find(query, projection), total=total_docs, desc="Processing articles"
    ):
        url = doc["url"]
        current_domain = get_domain(url)

        # Skip if domain is in ignored list
        if current_domain in ignored_domains:
            print(f"Skipping {url} (domain ignored due to multiple failures)")
            continue

        # If the domain is the same as the previous request, add a delay
        if current_domain == previous_domain:
            time.sleep(random.uniform(0.5, 2))

        content = fetch_one_article(url)

        if content:
            result = collection.update_one(
                {"_id": doc["_id"]}, {"$set": {"page_content": content}}
            )
            print(f"Updated {url}")
            # Reset error count for successful fetch
            domain_errors[current_domain] = 0
        else:
            print(f"No content fetched for {url}")
            domain_errors[current_domain] += 1

            # Check if domain should be ignored
            if domain_errors[current_domain] >= 3:
                ignored_domains.add(current_domain)
                print(f"Domain {current_domain} added to ignored list after 3 failures")

        previous_domain = current_domain

    print(f"Domains ignored due to multiple failures: {ignored_domains}")
    return ignored_domains


# Main execution
try:
    newly_ignored_domains = process_articles()

    # Optionally, update NOT_FETCHABLE_DOMAINS with newly ignored domains
    NOT_FETCHABLE_DOMAINS.extend(newly_ignored_domains)
    print(f"Updated NOT_FETCHABLE_DOMAINS: {NOT_FETCHABLE_DOMAINS}")

except KeyboardInterrupt:
    print("Process interrupted by user. Progress saved.")


# Update a CSV file


In [ ]:
%pip install -q pandas

import pandas as pd


### Load data from CSV


In [38]:
FILE = r"data\blogdb.ai_news_2024_07_02_no_embedding.csv"

df = pd.read_csv(FILE)

df.head()


,_id,date,title,body,url,image,source,found_at,page_content
0,667d1ef6fc45eb48396a6a0c,2024-06-20T14:00:00.000Z,Anthropic's rivalry with OpenAI heats up with ...,Just a month after OpenAI rolled out its lates...,https://www.msn.com/en-us/news/technology/anth...,https://img-s-msn-com.akamaized.net/tenant/amp...,Fortune on MSN.com,2024-06-26T09:51:56.553Z,NaN
1,667d1ef6fc45eb48396a6a07,2024-06-25T12:00:00.000Z,Etched is building an AI chip that only runs o...,"The transformer, proposed by a team of Google ...",https://techcrunch.com/2024/06/25/etched-is-bu...,https://techcrunch.com/wp-content/uploads/2023...,TechCrunch,2024-06-26T09:51:56.553Z,As generative AI touches a growing number of i...
2,667d1ef6fc45eb48396a6a06,2024-06-25T21:49:00.000Z,Boston scientists create AI model to 'catch Al...,Researchers say they've created a promising AI...,https://www.msn.com/en-us/health/other/boston-...,https://img-s-msn-com.akamaized.net/tenant/amp...,Tribune News Service on MSN.com,2024-06-26T09:51:56.553Z,NaN
3,667d1ef6fc45eb48396a6a0f,2024-06-25T18:07:00.000Z,FDA clears new AI-powered 12-lead ECG from Ali...,AliveCor announced today that it received FDA ...,https://www.massdevice.com/fda-clears-ai-12-le...,https://www.massdevice.com/wp-content/uploads/...,MassDevice,2024-06-26T09:51:56.553Z,AliveCor announced today that it received FDA ...
4,667d1ef6fc45eb48396a6a11,2024-06-26T08:00:00.000Z,EasyTranslate thinks augmenting LLMs with huma...,But now it's headed in a new direction with a ...,https://techcrunch.com/2024/06/26/easytranslat...,NaN,TechCrunch,2024-06-26T09:51:56.553Z,You might think new generative AI startups lik...


### Add `domain` field


In [39]:
df["domain"] = df["url"].apply(get_domain)

df.head()


,_id,date,title,body,url,image,source,found_at,page_content,domain
0,667d1ef6fc45eb48396a6a0c,2024-06-20T14:00:00.000Z,Anthropic's rivalry with OpenAI heats up with ...,Just a month after OpenAI rolled out its lates...,https://www.msn.com/en-us/news/technology/anth...,https://img-s-msn-com.akamaized.net/tenant/amp...,Fortune on MSN.com,2024-06-26T09:51:56.553Z,NaN,msn.com
1,667d1ef6fc45eb48396a6a07,2024-06-25T12:00:00.000Z,Etched is building an AI chip that only runs o...,"The transformer, proposed by a team of Google ...",https://techcrunch.com/2024/06/25/etched-is-bu...,https://techcrunch.com/wp-content/uploads/2023...,TechCrunch,2024-06-26T09:51:56.553Z,As generative AI touches a growing number of i...,techcrunch.com
2,667d1ef6fc45eb48396a6a06,2024-06-25T21:49:00.000Z,Boston scientists create AI model to 'catch Al...,Researchers say they've created a promising AI...,https://www.msn.com/en-us/health/other/boston-...,https://img-s-msn-com.akamaized.net/tenant/amp...,Tribune News Service on MSN.com,2024-06-26T09:51:56.553Z,NaN,msn.com
3,667d1ef6fc45eb48396a6a0f,2024-06-25T18:07:00.000Z,FDA clears new AI-powered 12-lead ECG from Ali...,AliveCor announced today that it received FDA ...,https://www.massdevice.com/fda-clears-ai-12-le...,https://www.massdevice.com/wp-content/uploads/...,MassDevice,2024-06-26T09:51:56.553Z,AliveCor announced today that it received FDA ...,massdevice.com
4,667d1ef6fc45eb48396a6a11,2024-06-26T08:00:00.000Z,EasyTranslate thinks augmenting LLMs with huma...,But now it's headed in a new direction with a ...,https://techcrunch.com/2024/06/26/easytranslat...,NaN,TechCrunch,2024-06-26T09:51:56.553Z,You might think new generative AI startups lik...,techcrunch.com


# Find not featchable domains


In [40]:
domain_popularity = df["domain"].value_counts()

domain_popularity.head(10)

domain
msn.com                        6851
finance.yahoo.com               867
yahoo.com                       667
forbes.com                      589
linkedin.com                    398
markets.businessinsider.com     369
tmcnet.com                      330
techcrunch.com                  233
businessinsider.com             213
lelezard.com                    192
Name: count, dtype: int64

In [41]:
one_article_per_domain = (
    df.sort_values("date", ascending=False)
    .drop_duplicates(subset="domain")
    .assign(domain_popularity=lambda x: x["domain"].map(domain_popularity))
    .sort_values("domain_popularity", ascending=False)
    .reset_index(drop=True)
)

one_article_per_domain.head(5)

,_id,date,title,body,url,image,source,found_at,page_content,domain,domain_popularity
0,6683b1f393fb337940318a80,2024-07-02T07:36:00.000Z,"OpenAI introduces CriticGPT, a GPT 4-based mod...","OpenAI has recently unveiled CriticGPT, which ...",https://www.msn.com/en-in/money/news/openai-in...,https://images.moneycontrol.com/static-mcnews/...,moneycontrol.com on MSN.com,2024-07-02T07:39:30.786Z,NaN,msn.com,6851
1,6683b1f393fb33794031890d,2024-07-02T07:29:00.000Z,(XS2491029380.SG),As the AI race continues to spearhead the tech...,https://finance.yahoo.com/quote/XS2491029380.S...,NaN,Yahoo Finance,2024-07-02T07:39:05.217Z,NaN,finance.yahoo.com,867
2,6683b1f393fb337940318ab8,2024-07-02T05:27:00.000Z,L.A. Metro safety concerns amid back-to-back s...,"In a span of just six and half hours, two Los ...",https://www.yahoo.com/news/l-metro-safety-conc...,https://s.yimg.com/ny/api/res/1.2/WsqLx4.5VqB3...,Yahoo,2024-07-02T07:39:35.204Z,NaN,yahoo.com,667
3,6683b1f393fb3379403189c5,2024-07-02T02:55:00.000Z,W San Francisco Partners With SIA Scotch Whisk...,"The W San Francisco hosted the annual ""What Sh...",https://www.forbes.com/sites/noelburgess/2024/...,https://imageio.forbes.com/specials-images/ima...,Forbes,2024-07-02T07:39:17.900Z,NaN,forbes.com,589
4,6683b1f393fb33794031925e,2024-07-02T00:31:00.000Z,Big tech eyes nuclear power for AI,The artificial intelligence boom has tech comp...,https://www.linkedin.com/news/story/big-tech-e...,https://media.licdn.com/dms/image/D4E1AAQFTU1I...,LinkedIn,2024-07-02T07:44:06.967Z,NaN,linkedin.com,398


### Fetch the articles content for the top domains to check if fetching works


In [42]:
from langchain_community.document_loaders import NewsURLLoader

TOP_K = 500

print(f"Will fetch one article for each of the top {TOP_K} domains")

urls = one_article_per_domain["url"].tolist()[:TOP_K]

loader = NewsURLLoader(urls, show_progress_bar=True, continue_on_failure=True)

data = []
# data = loader.load()

Will fetch one article for each of the top 500 domains


In [43]:
for doc in [doc for doc in data if doc.page_content][:10]:
    print(f"Title: {doc.metadata['title']}")
    print(f"URL: {doc.metadata['link']}")
    print(f"Text: {doc.page_content}...")  # Print first 100 characters
    print("---")
    print()

In [44]:
empty_page_content = [x.metadata["link"] for x in data if not x.page_content]

empty_page_content

[]

In [45]:
# failed_urls = list(
#     set(urls) - set(x.metadata["link"] for x in data) | set(empty_page_content)
# )
# failed_domains = [get_domain(url) for url in failed_urls]

# failed_domains[:10]



In [46]:
failed_domains = {
    "skift.com",
    "247wallst.com",
    "windowsreport.com",
    "news-medical.net",
    "heise.de",
    "news18.com",
    "khaleejtimes.com",
    "venturebeat.com",
    "dbta.com",
    "autoevolution.com",
    "finbold.com",
    "techstory.in",
    "ca.finance.yahoo.com",
    "hollywoodreporter.com",
    "pc-tablet.com",
    "cmswire.com",
    "bbc.co.uk",
    "barrons.com",
    "marktechpost.com",
    "bbc.com",
    "businesstoday.in",
    "bgr.com",
    "wfmz.com",
    "geeky-gadgets.com",
    "mobileworldlive.com",
    "latimes.com",
    "philstar.com",
    "9to5mac.com",
    "cio.economictimes.indiatimes.com",
    "dqindia.com",
    "law.com",
    "androidauthority.com",
    "telecoms.com",
    "variety.com",
    "securityboulevard.com",
    "securitysystemsnews.com",
    "uk.pcmag.com",
    "morningstar.com",
    "bworldonline.com",
    "windowscentral.com",
    "djournal.com",
    "marketplace.org",
    "thefastmode.com",
    "baltimoresun.com",
    "crn.com",
    "punchng.com",
    "foxbusiness.com",
    "indiaeducationdiary.in",
    "usnews.com",
    "infoworld.com",
    "theconversation.com",
    "moneycontrol.com",
    "indianexpress.com",
    "microsoft.com",
    "arabianbusiness.com",
    "finanznachrichten.de",
    "news.yahoo.com",
    "livemint.com",
    "yourstory.com",
    "onrec.com",
    "inc42.com",
    "financialit.net",
    "techopedia.com",
    "tech-critter.com",
    "fxstreet.com",
    "computerworld.com",
    "mybroadband.co.za",
    "crowdfundinsider.com",
    "uk.news.yahoo.com",
    "abc.net.au",
    "infoq.com",
    "fiercehealthcare.com",
    "bloomberg.com",
    "benzinga.com",
    "telegraph.co.uk",
    "eagletribune.com",
    "futurism.com",
    "menafn.com",
    "adage.com",
    "semiengineering.com",
    "artificiallawyer.com",
    "wired.com",
    "tweaktown.com",
    "campustechnology.com",
    "sports.yahoo.com",
    "neowin.net",
    "hbr.org",
    "insidebigdata.com",
    "bernama.com",
    "medianama.com",
    "financialpost.com",
    "twinfinite.net",
    "yicaiglobal.com",
    "thehindu.com",
    "rcrwireless.com",
    "computerweekly.com",
    "executivegov.com",
    "businesstimes.com.sg",
    "cnet.com",
    "ciodive.com",
    "es-us.finanzas.yahoo.com",
    "pcmag.com",
    "phonearena.com",
    "pbs.org",
    "gizchina.com",
    "statnews.com",
    "businessghana.com",
    "global.chinadaily.com.cn",
    "businessinsider.in",
    "gizmodo.com",
    "pocket-lint.com",
    "sciencedaily.com",
    "csoonline.com",
    "straitstimes.com",
    "theedgesingapore.com",
    "news.crunchbase.com",
    "knowyourmeme.com",
    "decrypt.co",
    "enterprisetimes.co.uk",
    "news.europawire.eu",
    "kelo.com",
    "pcgamesn.com",
    "manilatimes.net",
    "thederrick.com",
    "itweb.co.za",
    "marketing-interactive.com",
    "nextgov.com",
    "fortune.com",
    "technode.com",
    "techradar.com",
    "livescience.com",
    "edition.cnn.com",
    "theregister.com",
    "tomsguide.com",
    "hackernoon.com",
    "valdostadailytimes.com",
    "ctvnews.ca",
    "businessinsider.com",
    "techspot.com",
    "sg.finance.yahoo.com",
    "thesangaiexpress.com",
    "telegraphindia.com",
    "euronews.com",
    "mirror.co.uk",
    "gulf-times.com",
    "au.pcmag.com",
    "lomeactu.com",
    "economictimes.indiatimes.com",
    "finance.yahoo.com",
    "federalnewsnetwork.com",
    "br.advfn.com",
    "deccanchronicle.com",
    "siliconindia.com",
    "ca.news.yahoo.com",
    "aibusiness.com",
    "scientificamerican.com",
    "ntnews.com.au",
    "eurekalert.org",
    "musicbusinessworldwide.com",
    "businessmirror.com.ph",
    "coingape.com",
    "sharecafe.com.au",
    "economist.com",
    "ign.com",
    "timesunion.com",
    "livebitcoinnews.com",
    "washingtonpost.com",
    "foxnews.com",
    "ippmedia.com",
    "hrdive.com",
    "pressdemocrat.com",
    "rediff.com",
    "beebom.com",
    "thurrott.com",
    "thewindowsclub.com",
    "informazione.it",
    "poynter.org",
    "searchengineland.com",
    "indiatoday.in",
    "eschoolnews.com",
    "digitaljournal.com",
    "stockhouse.com",
    "bleedingcool.com",
    "androguider.com",
    "infosecurity-magazine.com",
    "scmagazine.com",
    "odishatv.in",
    "livewiremarkets.com",
    "securityinfowatch.com",
    "cio.com",
    "pcworld.com",
    "au.finance.yahoo.com",
    "redmondmag.com",
    "cbc.ca",
    "insidebitcoins.com",
    "nbclosangeles.com",
    "fool.com",
    "wsj.com",
    "billboard.com",
    "sg.news.yahoo.com",
    "neurosciencenews.com",
    "sharewise.com",
    "ibtimes.com",
    "forbes.com",
    "nextbigfuture.com",
    "news.webindia123.com",
    "unite.ai",
    "energycentral.com",
    "nzherald.co.nz",
    "newsbytesapp.com",
    "digitaltrends.com",
    "thisdaylive.com",
    "mensjournal.com",
    "beincrypto.com",
    "news.sky.com",
    "breakingdefense.com",
    "tradearabia.com",
    "thenextweb.com",
    "destinationcrm.com",
    "genengnews.com",
    "therobotreport.com",
    "mumbrella.com.au",
    "hoodline.com",
    "techcrunch.com",
    "kahawatungu.com",
    "nasdaq.com",
    "gigazine.net",
    "itnews.com.au",
    "apnews.com",
    "techrepublic.com",
    "sdtimes.com",
    "iblnews.org",
    "investopedia.com",
    "cbsnews.com",
    "technical.ly",
    "seekingalpha.com",
    "inverse.com",
    "geekwire.com",
    "leewayhertz.com",
    "diginomica.com",
    "fmiblog.com",
    "itwire.com",
    "outlookindia.com",
    "cybernews.com",
    "hindustantimes.com",
    "koreaherald.com",
    "aastocks.com",
    "finextra.com",
    "thehackernews.com",
    "techrights.org",
    "techxplore.com",
    "victoriaadvocate.com",
    "slator.com",
    "dallasinnovates.com",
    "saipantribune.com",
    "miamiherald.com",
    "cnn.com",
    "timesofindia.indiatimes.com",
    "komando.com",
    "cyprus-mail.com",
    "travelweekly.com.au",
    "gizmodo.com.au",
    "lablab.ai",
    "datacenterdynamics.com",
    "observer.com",
    "digit.in",
    "firstpost.com",
    "prnewswire.co.uk",
    "devdiscourse.com",
    "animenewsnetwork.com",
    "darkreading.com",
    "arstechnica.com",
    "ft.com",
    "sourcesecurity.com",
    "frontiersin.org",
    "guru3d.com",
    "coinspeaker.com",
    "thewrap.com",
    "telecom.economictimes.indiatimes.com",
    "bangkokpost.com",
    "channelnewsasia.com",
    "news.bloomberglaw.com",
    "technology.inquirer.net",
    "deadline.com",
    "htxt.co.za",
    "joplinglobe.com",
    "thedrum.com",
    "scoop.co.nz",
    "fastcompany.com",
    "technologynetworks.com",
    "newscientist.com",
    "en.globes.co.il",
    "thehill.com",
    "filmibeat.com",
    "time.com",
    "mobilemarketingmagazine.com",
    "medscape.com",
    "infotechlead.com",
    "biometricupdate.com",
    "theatlantic.com",
    "thehindubusinessline.com",
    "hackaday.com",
    "theprint.in",
    "eurogamer.net",
    "businesswire.com",
    "money.rediff.com",
    "indiatvnews.com",
    "cacm.acm.org",
    "ndtv.com",
    "it-online.co.za",
    "news.marketersmedia.com",
    "asia.nikkei.com",
    "tvbeurope.com",
    "govtech.com",
    "jpost.com",
    "daytondailynews.com",
    "vietbao.vn",
    "linkedin.com",
    "technologyreview.com",
    "dailytelegraph.com.au",
    "cryptobriefing.com",
    "advanced-television.com",
    "pocketgamer.com",
    "cryptopolitan.com",
    "gamespot.com",
    "ibtimes.co.uk",
    "visualstudiomagazine.com",
    "cnbctv18.com",
    "vir.com.vn",
    "nypost.com",
    "entrepreneur.com",
    "freepressjournal.in",
    "thenews-chronicle.com",
    "theglobeandmail.com",
    "gizbot.com",
    "news.microsoft.com",
    "singularityhub.com",
    "networkworld.com",
    "techbullion.com",
    "thestar.com.my",
    "electronicdesign.com",
    "cambridge.org",
    "ocbj.com",
    "beckershospitalreview.com",
    "gadgets360.com",
    "smh.com.au",
    "popsci.com",
    "electronics360.globalspec.com",
    "nbcnews.com",
    "markets.businessinsider.com",
    "nytimes.com",
    "techzine.eu",
    "albawaba.com",
    "asiaone.com",
    "financialexpress.com",
    "chicagotribune.com",
    "thejournal.com",
    "theinformation.com",
    "campaignasia.com",
    "uk.investing.com",
    "siliconrepublic.com",
    "insidehpc.com",
    "designnews.com",
    "daijiworld.com",
    "au.news.yahoo.com",
    "insidermonkey.com",
    "in.mashable.com",
    "thehansindia.com",
    "chinadaily.com.cn",
    "fudzilla.com",
    "scitechdaily.com",
    "bignewsnetwork.com",
    "irishtimes.com",
    "brandequity.economictimes.indiatimes.com",
    "phys.org",
    "tradersmagazine.com",
    "goshennews.com",
    "analyticsindiamag.com",
    "appleinsider.com",
    "gazette.com",
    "hcamag.com",
    "adexchanger.com",
    "aboutamazon.com",
    "thesun.co.uk",
    "9to5google.com",
    "digitimes.com",
    "news.com.au",
    "business-standard.com",
    "nbcnewyork.com",
    "uk.finance.yahoo.com",
    "fbcnews.com.fj",
    "edweek.org",
    "crypto-news-flash.com",
    "deccanherald.com",
    "tmcnet.com",
    "china.org.cn",
    "politico.com",
    "newswit.com",
    "gulfnews.com",
    "dexerto.com",
    "vox.com",
    "dailydot.com",
    "lifehacker.com",
    "bizjournals.com",
    "lelezard.com",
    "dailymail.co.uk",
    "androidpolice.com",
    "medindia.net",
    "dailyhodl.com",
    "cryptonews.com",
    "dcvelocity.com",
    "searchenginejournal.com",
    "buffalo.edu",
    "macrumors.com",
    "cointelegraph.com",
    "inc.com",
    "jdsupra.com",
    "gamerant.com",
    "mmm-online.com",
    "fiverr.com",
    "newsweek.com",
    "pionline.com",
    "syncedreview.com",
    "mg.co.za",
    "psychologytoday.com",
    "crn.com.au",
    "stuff.tv",
    "couriermail.com.au",
    "tribune.com.pk",
    "scmp.com",
    "globaltimes.cn",
    "androidheadlines.com",
    "nocamels.com",
    "playtoearngames.com",
    "theverge.com",
    "english.aawsat.com",
    "malaysiasun.com",
    "techreport.com",
    "newatlas.com",
    "hothardware.com",
    "bleepingcomputer.com",
    "tbsnews.net",
    "sdxcentral.com",
    "investing.com",
    "independent.co.uk",
    "bbntimes.com",
    "msn.com",
    "tribuneindia.com",
    "abcnews.go.com",
    "securityweek.com",
    "newindianexpress.com",
    "design-reuse.com",
    "techtimes.com",
    "marketwatch.com",
    "bandt.com.au",
    "taiwannews.com.tw",
    "gizmochina.com",
    "rappler.com",
    "channelnews.com.au",
    "zawya.com",
    "washingtontimes.com",
    "fonearena.com",
    "fedscoop.com",
    "dailymemphian.com",
    "news.stocktradersdaily.com",
    "betakit.com",
    "yahoo.com",
    "themalaysianreserve.com",
    "zdnet.com",
    "afr.com",
    "israel21c.org",
    "siliconangle.com",
    "datanami.com",
    "seattletimes.com",
    "digitalinformationworld.com",
    "koreatimes.co.kr",
    "standard.co.uk",
    "sfgate.com",
    "tech.hindustantimes.com",
    "accountingtoday.com",
    "americanbanker.com",
    "econotimes.com",
    "bostonglobe.com",
    "01net.it",
    "money.usnews.com",
    "nature.com",
    "thestreet.com",
    "campaignlive.co.uk",
    "medicalxpress.com",
    "zeebiz.com",
    "mashable.com",
    "taipeitimes.com",
    "usatoday.com",
    "manilastandard.net",
    "defenseone.com",
    "insidehighered.com",
    "wgnradio.com",
    "smartcompany.com.au",
    "theaustralian.com.au",
    "reuters.com",
}

In [47]:
for doc in [doc for doc in data if doc.page_content][:2]:
    print(f"Title: {doc.metadata['title']}")
    print(f"URL: {doc.metadata['link']}")
    print(f"Text: {doc.page_content}...")  # Print first 100 characters
    print("---")
    print()

# Fetch the missing articles using `newspaper3k`


In [51]:
import time
import random
from newspaper import Article
from tqdm import tqdm
import pandas as pd
from collections import defaultdict


def process_articles(
    df, limit: int | None = None, failed_domains: set[str] | None = None, max_failures=3
):
    """
    Process articles from a DataFrame, fetching content for each article URL.

    This function iterates through the provided DataFrame, attempting to fetch
    the content for each article URL. It respects rate limiting, skips articles
    from failed domains, stops processing after reaching the specified limit,
    and tracks new domains that consistently fail.

    Parameters:
    -----------
    df : pandas.DataFrame
        A DataFrame containing article information. Must include 'url' and
        'page_content' columns.
    limit : int, optional
        The maximum number of articles to process. If None, all articles are processed.
    failed_domains : set, optional
        A set of domain names to skip (e.g., {'example.com', 'faildomain.com'}).
    max_failures : int, optional
        The number of failures allowed for a domain before it's added to ignored_domains.

    Returns:
    --------
    tuple
        A tuple containing:
        - The updated DataFrame with fetched content in the 'page_content' column.
        - A set of newly ignored domains.

    Side Effects:
    -------------
    - Updates the input DataFrame in-place.
    """
    failed_domains = failed_domains or set()
    new_failed_domains = set()
    domain_failures = defaultdict(int)

    previous_domain = None
    updated_count = 0
    skipped_count = 0
    processed_count = 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Processing articles"):
        if limit is not None and processed_count >= limit:
            print(f"Reached the limit of {limit} articles. Stopping.")
            break

        url = row["url"]
        current_domain = get_domain(url)

        # Skip if the domain is in the failed_domains or ignored_domains list
        if current_domain in failed_domains or current_domain in new_failed_domains:
            # print(f"Skipping {url} (domain failing)")
            skipped_count += 1
            continue

        # Skip if content already exists
        if pd.notna(row["page_content"]):
            # print(f"Content already exists for {url}")
            skipped_count += 1
            continue

        # If the domain is the same as the previous request, add a delay
        if current_domain == previous_domain:
            time.sleep(random.uniform(1, 3))

        content = fetch_one_article(url)

        if content:
            # Update the DataFrame
            df.at[_, "page_content"] = content
            print(f"Updated content for {url}")
            updated_count += 1
            # Reset failure count for successful fetch
            domain_failures[current_domain] = 0
        else:
            print(f"No content fetched for {url}")
            skipped_count += 1
            # Increment failure count for the domain
            domain_failures[current_domain] += 1

            # Check if domain should be ignored
            if domain_failures[current_domain] >= max_failures:
                new_failed_domains.add(current_domain)
                print(
                    f"Added {current_domain} to ignored domains after {max_failures} failures"
                )

        previous_domain = current_domain
        processed_count += 1

    print(f"Updated {updated_count} articles, skipped {skipped_count} articles")
    print(f"Total processed: {processed_count}")
    print(f"Newly ignored domains: {new_failed_domains}")
    return df, new_failed_domains


# Example usage
try:
    df, new_ignored_domains = process_articles(
        df, limit=None, failed_domains=set(failed_domains)
    )

    # Update failed_domains with newly ignored domains
    failed_domains.update(new_ignored_domains)

except KeyboardInterrupt:
    print("Process interrupted by user. Progress saved.")

Processing articles:  97%|█████████▋| 25451/26106 [1:07:15<06:03,  1.80it/s]

Updated content for https://ecommercefastlane.com/how-to-create-a-digital-marketing-package/


Processing articles:  98%|█████████▊| 25465/26106 [1:07:16<02:24,  4.44it/s]

Updated content for https://kval.com/news/local/wildfire-season-prepare-protect-kval-preparing-you-for-the-upcoming-wildfire-season


Processing articles:  98%|█████████▊| 25471/26106 [1:07:17<02:24,  4.41it/s]

...

Updated content for https://artreview.com/the-top-10-exhibitions-to-see-in-july-2024/


Processing articles: 100%|█████████▉| 26089/26106 [1:09:47<00:05,  3.35it/s]

Updated content for https://www.examenglish.com/FCE/fce_use_of_english_part2.htm


Processing articles: 100%|██████████| 26106/26106 [1:09:47<00:00,  6.23it/s]

Updated content for https://www.sportspromedia.com/news/the-open-ntt-data-tech-5g-digital-twin/
Updated 2890 articles, skipped 23216 articles
Total processed: 3512
Newly ignored domains: {'charlotteobserver.com', 'standard-journal.com', 'weeklytimesnow.com.au', 'therepublic.com', 'autonews.com', 'kfor.com', 'sinchew.com.my', 'bozemandailychronicle.com', 'agoramagazine.it', 'business-reporter.co.uk', 'qmul.ac.uk', 'caledonianrecord.com', 'simpleflying.com', 'thelancet.com', 'triblive.com', 'chron.com', 'thegrio.com', 'macaubusiness.com', 'in.investing.com', 'greekreporter.com', 'realclearscience.com', 'sbr.com.sg', 'extremetech.com', 'kdvr.com', 'chronicle.com', 'kpvi.com', 'yorkshirepost.co.uk', 'purdueexponent.org', 'insidermedia.com', 'ijr.com', 'kcci.com', 'marketindex.com.au', 'moneymorning.com', 'blogs.timesofisrael.com', 'scotsman.com', 'startupdaily.net', 'agi.it', 'news-journal.com', 'citizen.co.za', 'iol.co.za', 'sfweekly.com', 'lse.co.uk', 'english.ahram.org.eg', 'khou.com', 

### Export updated df


In [55]:
from datetime import datetime
from os.path import join

output = join("data", f"blogdb.ai_news_{datetime.now().strftime('%Y_%m_%d_%H_%M')}.csv")

df.to_csv(output, index=False, encoding="utf-8")

# Update mongoDB with the page_content
